In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB2
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

data_dir = "/kaggle/input/cell-images-for-detecting-malaria/cell_images"
print("Data dir:", data_dir)
print("Classes:", os.listdir(data_dir))


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Use only the class folders, ignore the extra "cell_images" inside
data_dir = "/kaggle/input/cell-images-for-detecting-malaria/cell_images"

image_paths = []
labels = []

for label_name in ["Parasitized", "Uninfected"]:
    class_dir = os.path.join(data_dir, label_name)
    for fname in os.listdir(class_dir):
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths.append(os.path.join(class_dir, fname))
            labels.append(1 if label_name == "Parasitized" else 0)

image_paths = np.array(image_paths)
labels = np.array(labels)

print("Total images:", len(image_paths))
print("Parasitized:", np.sum(labels == 1))
print("Uninfected:", np.sum(labels == 0))

# 80/20 split
train_paths, test_paths, train_labels, test_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

print("Train size:", len(train_paths))
print("Test size:", len(test_paths))


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = (260, 260)
batch_size = 32

# Map numeric labels back to string class names for flow_from_dataframe
train_df = pd.DataFrame({
    "filepath": train_paths,
    "label": np.where(train_labels == 1, "Parasitized", "Uninfected")
})

test_df = pd.DataFrame({
    "filepath": test_paths,
    "label": np.where(test_labels == 1, "Parasitized", "Uninfected")
})

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.1
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    train_df,
    x_col="filepath",
    y_col="label",
    target_size=img_size,
    class_mode="binary",
    batch_size=batch_size,
    subset="training",
    shuffle=True
)

val_gen = train_datagen.flow_from_dataframe(
    train_df,
    x_col="filepath",
    y_col="label",
    target_size=img_size,
    class_mode="binary",
    batch_size=batch_size,
    subset="validation",
    shuffle=False
)

test_gen = test_datagen.flow_from_dataframe(
    test_df,
    x_col="filepath",
    y_col="label",
    target_size=img_size,
    class_mode="binary",
    batch_size=batch_size,
    shuffle=False
)


In [ ]:
import time
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB2

input_shape = (img_size[0], img_size[1], 3)

# Base EfficientNet-B2 model
base_model = EfficientNetB2(
    include_top=False,
    weights="imagenet",
    input_shape=input_shape
)

base_model.trainable = True  # fine-tune all layers

# Custom head (matches paper structure)
inputs = layers.Input(shape=input_shape)
x = base_model(inputs, training=True)
x = layers.Dropout(0.3)(x)
x = layers.Flatten()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Dropout(0.5)(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(32)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )
]

start_time = time.time()

history = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks
)

end_time = time.time()
print(f"Training time: {(end_time - start_time)/60:.2f} minutes")


In [ ]:
test_loss, test_acc, test_auc, test_prec, test_rec = model.evaluate(test_gen)
test_f1 = 2 * test_prec * test_rec / (test_prec + test_rec)

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)
print("Test AUC:", test_auc)
print("Test precision:", test_prec)
print("Test recall:", test_rec)
print("Test F1:", test_f1)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
import numpy as np
import pandas as pd

# ==========================================
# GET PREDICTIONS (if not already done)
# ==========================================
test_gen.reset()
preds = model.predict(test_gen, verbose=0)
y_pred = (preds > 0.5).astype(int).flatten()
y_true = test_gen.classes

# ==========================================
# 1. CONFUSION MATRIX
# ==========================================
print("=" * 60)
print("1. CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=["Uninfected", "Parasitized"], 
            yticklabels=["Uninfected", "Parasitized"],
            cbar_kws={"label": "Count"})
plt.title("Confusion Matrix - Test Set Evaluation", fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: confusion_matrix.png")
plt.show()

# ==========================================
# 2. CLASSIFICATION REPORT
# ==========================================
print("\n" + "=" * 60)
print("2. CLASSIFICATION REPORT")
print("=" * 60)

print(classification_report(y_true, y_pred, 
                           target_names=["Uninfected", "Parasitized"]))

report_dict = classification_report(y_true, y_pred, 
                                    target_names=["Uninfected", "Parasitized"],
                                    output_dict=True)

# ==========================================
# 3. ROC CURVE
# ==========================================
print("\n" + "=" * 60)
print("3. ROC CURVE")
print("=" * 60)

fpr, tpr, _ = roc_curve(y_true, preds)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2.5, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Malaria Detection', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300, bbox_inches='tight')
print(f"AUC Score: {roc_auc:.4f}")
print("✓ Saved: roc_curve.png")
plt.show()

# ==========================================
# 4. PRECISION-RECALL CURVE
# ==========================================
print("\n" + "=" * 60)
print("4. PRECISION-RECALL CURVE")
print("=" * 60)

precision, recall, _ = precision_recall_curve(y_true, preds)
pr_auc = auc(recall, precision)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2.5, label=f'PR Curve (AUC = {pr_auc:.4f})')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve - Malaria Detection', fontsize=14, fontweight='bold')
plt.legend(loc="best", fontsize=11)
plt.grid(alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.tight_layout()
plt.savefig('precision_recall_curve.png', dpi=300, bbox_inches='tight')
print(f"PR-AUC Score: {pr_auc:.4f}")
print("✓ Saved: precision_recall_curve.png")
plt.show()

# ==========================================
# 5. TRAINING HISTORY
# ==========================================
print("\n" + "=" * 60)
print("5. TRAINING HISTORY PLOTS")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0, 0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('Accuracy', fontsize=11)
axes[0, 0].set_title('Model Accuracy Over Epochs', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0, 1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel('Loss', fontsize=11)
axes[0, 1].set_title('Model Loss Over Epochs', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(history.history['auc'], label='Train AUC', linewidth=2)
axes[1, 0].plot(history.history['val_auc'], label='Validation AUC', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=11)
axes[1, 0].set_ylabel('AUC', fontsize=11)
axes[1, 0].set_title('Model AUC Over Epochs', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(history.history['precision'], label='Train Precision', linewidth=2)
axes[1, 1].plot(history.history['recall'], label='Train Recall', linewidth=2)
axes[1, 1].plot(history.history['val_precision'], label='Val Precision', linewidth=2, linestyle='--')
axes[1, 1].plot(history.history['val_recall'], label='Val Recall', linewidth=2, linestyle='--')
axes[1, 1].set_xlabel('Epoch', fontsize=11)
axes[1, 1].set_ylabel('Score', fontsize=11)
axes[1, 1].set_title('Precision & Recall Over Epochs', fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
print("✓ Saved: training_history.png")
plt.show()

# ==========================================
# 6. SUMMARY TABLE
# ==========================================
print("\n" + "=" * 60)
print("6. RESULTS SUMMARY TABLE")
print("=" * 60)

summary_data = {
    "Class": ["Uninfected", "Parasitized", "Weighted Avg"],
    "Precision": [
        f"{report_dict['Uninfected']['precision']:.4f}",
        f"{report_dict['Parasitized']['precision']:.4f}",
        f"{report_dict['weighted avg']['precision']:.4f}"
    ],
    "Recall": [
        f"{report_dict['Uninfected']['recall']:.4f}",
        f"{report_dict['Parasitized']['recall']:.4f}",
        f"{report_dict['weighted avg']['recall']:.4f}"
    ],
    "F1-Score": [
        f"{report_dict['Uninfected']['f1-score']:.4f}",
        f"{report_dict['Parasitized']['f1-score']:.4f}",
        f"{report_dict['weighted avg']['f1-score']:.4f}"
    ],
    "Support": [
        f"{int(report_dict['Uninfected']['support'])}",
        f"{int(report_dict['Parasitized']['support'])}",
        f"{int(report_dict['weighted avg']['support'])}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))
summary_df.to_csv('test_results_summary.csv', index=False)
print("\n✓ Saved: test_results_summary.csv")

print("\n" + "=" * 60)
print("✓ ALL IMAGES & TABLES GENERATED")
print("=" * 60)
print("\nFiles saved:")
print("  • confusion_matrix.png")
print("  • roc_curve.png")
print("  • precision_recall_curve.png")
print("  • training_history.png")
print("  • test_results_summary.csv")
